# **Task 2: Filtering, Classifying, and Analyzing Vegetation with NDVI**

In this exercise, we will build upon what we have learned in Task 1 and explore how vegetation changes over time using **Google Earth Engine (GEE)**.  

You will learn to work with multi-temporal satellite imagery to build and visualize NDVI time series, and to summarize vegetation conditions across administrative units such as counties. This combined **spatio-temporal analysis** helps reveal patterns in ecosystem dynamics, agricultural productivity, and land-use change.


We will:
- Filter a **Sentinel-2 Surface Reflectance** or collection by location, date, and cloud percentage.  
- Compute the **Normalized Difference Vegetation Index (NDVI)** = (NIR − Red) / (NIR + Red).  
- Aggregate the NDVI values over time (e.g., monthly or yearly averages).  
- Visualize NDVI trends as charts to interpret vegetation patterns.
- Perform **zonal statistics** to summarize NDVI at the county level.    
- Optionally, export results (e.g., NDVI composites or charts) for reporting.

This task introduces **time series analysis in Earth Engine**, showing how cloud filtering, compositing, and visualization techniques can reveal temporal trends in vegetation health.


## **Initial Setup**

We need to import the necessary Python package for this task. We also need to authenticate and initialize the Earth Engine package with our credentials.

If you don't have your credentials yet, please refer to the "***Registration to Earth Engine - Step by Step guide***" PDF on NTU Learn

In [2]:
import ee
import geemap

ee.Authenticate()
ee.Initialize(project="YOUR_PROJECTID") # Replace with your project's ID


### **Step 1. Load the Map interface and Define an Area of Interest**

In [3]:
# Helper to make a fresh map with Google Hybrid basemap
m = geemap.Map(center=[41.9, -93.9], zoom=7)

m.add_tile_layer(
    url="https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}",
    name="Google Hybrid",
    attribution="Google"
)

m

Map(center=[41.9, -93.9], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright',…

In [8]:
# A) Option 1: Use the drawn AOI from the map (Polygon/Rectangle)
aoi = m.user_roi  # returns only the last geometry you drew

# Check if an AOI was drawn
if aoi is None:
    print('No AOI was found.')
    print('Please draw a polygon or rectangle on the map using the drawing tools and re-run this cell.')
else:
    # B) Option 2: Fallback AOI (Iowa-ish bounding box)
    # If you prefer to use a predefined area, uncomment the line below and comment out the 'aoi = m.user_roi' line above.
    # aoi = ee.Geometry.Rectangle([-94.5, 41.5, -93.5, 42.5])

    print("AOI ready. You can proceed to the next step")

AOI ready. You can proceed to the next step


### **Step 2. Load & filter Sentinel-2 (clouds < 5%)**

We will be filtering the **Sentinel-2 Surface Reflectance (SR) Harmonized collection**.

The filtering we will perform will ensure that we only keep scenes that:
- Intersect the **Area of Interest (AOI)**, AND  
- Were acquired within the specified **date range**, AND  
- Have a cloud cover percentage of **less than 5%**








In [9]:
# --- Define the date range of interest ---
# Here we specify the start and end dates for the time period we want to analyze.
start = "2016-01-01"
end   = "2024-12-31"

# --- Load and filter the Sentinel-2 Surface Reflectance image collection ---
s2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")  # Load the Sentinel-2 SR collection
    .filterBounds(aoi)                                  # Keep only images that intersect our area of interest (AOI)
    .filterDate(start, end)                             # Keep images acquired within the defined time range
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 5)) # Keep only images with less than 5% cloud cover
)

# --- Print the number of available images after filtering ---
print("Images in collection:", s2.size().getInfo())

Images in collection: 490


### **Step 3. Add NDVI band to each image**

We have already covered the topic of *Spectral Indices* in this course. Now we will utilize the filtered collection you created above and calculate the *Normalized Difference Vegetation Index* **(NDVI)** for each image.

Recall that NDVI is one of the most widely used vegetation indices in remote sensing because it highlights **live green vegetation**.  

- **High NDVI values (~0.6–0.9)** usually indicate dense, healthy vegetation (e.g., forests, crops in peak growing season).  
- **Moderate values (~0.2–0.5)** often correspond to shrubs, grasslands, or sparse vegetation.  
- **Low or negative values (≤ 0.1)** usually indicate bare soil, urban areas, water, or clouds.  

Remember that NDVI is calculated as:


$$
NDVI = \frac{NIR - RED}{NIR + RED}
$$


In [10]:
# Define the function that computes ndvi for each image
def add_ndvi(img):
    ndvi = img.normalizedDifference(["B8", "B4"]).rename("NDVI") # Using 'normalizedDifference' function, a native EE function
    return img.addBands(ndvi)

s2_ndvi = s2.map(add_ndvi).select("NDVI") # 'map' function allows us to apply the same function to each ee.Image in the collection
print("Prepared NDVI collection. Proceed to the next step")

Prepared NDVI collection. Proceed to the next step


### **Step 4. Apply Reductions**

Once the NDVI has been calculated for each scene, we can proceed to perform some **reduction operations** and **time series analysis**.  

Reduction operations are used to **summarize multiple images or values** into a single result. This is especially important when dealing with an ImageCollection spanning many dates.  

Some common reduction operations include:  

* **mean** – average value across all images (smooths out variability)  
* **median** – middle value across images (robust to outliers such as clouds or haze)  
* **min** – minimum value (useful for detecting bare soil or drought conditions)  
* **max** – maximum value (useful for peak vegetation conditions)  
* **sum** – total across all images (less common for NDVI, but useful for other bands like precipitation)  

In this exercise, we will use the **per-pixel `median`** for the NDVI band.  
This gives us a single composite image that represents the typical NDVI value for the year, while reducing the influence of noise and residual cloud contamination.  
Finally, we will **clip** the result so that the composite only covers our chosen AOI.  


In [11]:
# Calculating the per-pixel NVDI median and clipping
median_ndvi = s2_ndvi.median().clip(aoi)

# Create a new map object
m2 = geemap.Map()
m2.add_basemap('SATELLITE')

# 1. Define a more descriptive, multi-color palette
ndvi_palette = [
    '#440154', '#414487', '#2A788E', '#22A884', '#7AD151', '#FDE725'
]

# 2. Define the visualization parameters with the new palette and your specified range
ndvi_vis = {
    "min": -1,
    "max": 1,
    "palette": ndvi_palette
}

# 3. Add the layers to the map
m2.add_layer(aoi, {"opacity": 0.1}, "AOI")
m2.addLayer(median_ndvi, ndvi_vis, "Median NDVI")

# Center the map and add the inspector
m2.centerObject(median_ndvi, zoom=8)
m2.add_inspector()

# --- Add a colorbar corresponding to the NDVI palette ---
m2.add_colorbar(
    vis_params=ndvi_vis,
    label="NDVI",
    layer_name="Median NDVI",
    orientation="horizontal"  # or "vertical"
)

m2

Map(center=[41.58491567547602, -93.6220540000006], controls=(WidgetControl(options=['position', 'transparent_b…

###  Questions for Reflection - Part A

You have now created a **median NDVI composite** clipped to your AOI. Take a moment to observe the map you just generated.  

- **What do you see?**  
  Describe the spatial patterns of vegetation in your AOI. Which areas appear greener (higher NDVI) and which appear less vegetated?  

- **Can you identify any patterns?**  
  Do you notice differences between urban, agricultural, and natural areas?

- **What might explain these patterns?**  
  Think about land cover, seasonality, or human activity. How do these factors relate to NDVI values?  

- **Limitations:**  
  Does using a median composite hide any short-term changes (e.g., crop cycles, seasonal variations)? How might this affect your interpretation?  


### **Step 5. NDVI Time Series Analysis**

So far, we have visualized the **median NDVI composite** for the year 2022 over our AOI. While this provides a useful snapshot of typical vegetation conditions, Earth Engine allows us to go further.  

One of the strengths of GEE is its ability to handle **temporal data at scale**. Instead of looking at a single summary image, we can examine how NDVI values **change over time** by constructing a **time series plot**.  

Time series analysis helps us:  
- Identify **seasonal vegetation dynamics** (e.g., crop cycles, wet vs. dry seasons).  
- Detect **long-term trends** in vegetation health.  
- Compare different years or regions in a consistent way.  



**In the next step**, we will generate a time series plot showing how the mean NDVI in our AOI varies throughout the calendar year. This will allow us to link the composite map with the **temporal dynamics** driving it.  

In [12]:
from geemap import chart

fig = chart.image_series(
    s2_ndvi,                 # Our ImageCollection
    region=aoi,
    reducer=ee.Reducer.mean(),
    scale=100,
    x_property="system:time_start",     # ⬅️ time on x-axis
    chart_type="LineChart",
    x_cols="date",
    y_cols=["NDVI"],
    title="NDVI Mean over AOI",
    x_label="Date",
    y_label="NDVI",
    maxPixels=1e13
)
fig


###  Questions for Reflection - Part B

You just created a time series of **mean NDVI values** across your AOI for 2022.
Notice how the curve rises during the growing season and falls during winter or dry months.  

However, you may also see **sudden spikes and dips** in the curve.  


- What factors could cause abrupt increases or decreases in NDVI?  
  *Hint: think about atmospheric effects, residual cloud contamination, or mixed land cover types within the AOI.*  
- How might the **frequency of satellite overpasses** influence the smoothness of the time series?  
- Are there **seasonal transitions** (e.g., planting, harvest, leaf-on/leaf-off) that naturally explain some of the variation?  
- If you were analyzing this data for agricultural or ecological monitoring, how would you **reduce noise** or make the time series more interpretable?  

### **Step 6. Integrating Zonal Statistics**

Up to this point, we have focused on analyzing NDVI over a single **Area of Interest (AOI)**.  

Now, we will scale up and perform a similar analysis at the **county level** across the United States, using geospatial boundary data already hosted in Earth Engine.  

For this purpose, we will use the **`reduceRegions`** operator.  
- `reduceRegions` allows us to apply a **reducer** (e.g., mean, median, max) to each feature in a vector dataset, summarizing raster values (NDVI) within its boundaries.  
- This is a powerful way to link **pixel-based raster data** with **administrative or thematic boundaries** such as counties, provinces, watersheds, or farm plots.  

In this step, we will calculate the **mean NDVI per county** and export the results for further analysis. Once you run the export, check the **"Tasks" tab** in the Earth Engine Code Editor, or open the [Earth Engine Tasks Dashboard](https://console.cloud.google.com/earth-engine/tasks) to monitor progress.   

**Note:**  
You are not limited to built-in datasets. You can also **upload your own vector data** (e.g., shapefiles, GeoJSON, KML) to Earth Engine and run the exact same type of analysis on your custom regions of interest.

In [13]:
# Counties for the US
counties = ee.FeatureCollection("TIGER/2018/Counties")

# We will filter the counties and keep only the ones intersecting our AOI
counties_aoi = counties.filterBounds(aoi)

#
stats_fc = median_ndvi.reduceRegions(
    collection=counties_aoi,
    reducer=ee.Reducer.mean(),
    scale=100
)

# --- Export to Google Drive as GeoJSON ---
geemap.ee_export_vector_to_drive(
    collection=stats_fc,
    description="NDVI_per_county",
    folder="NDVI_exports",
    fileFormat="GeoJSON"
)

Exporting NDVI_per_county... Please check the Task Manager from the JavaScript Code Editor.


### **Step 7. Visualizing County-Level NDVI**  

Now that we have calculated the **mean NDVI per county**, we can bring those results back into a map for visualization.  

To do this, we will:  
1. Convert the FeatureCollection of county statistics into a raster image using `reduceToImage`.  
2. Define a visualization palette to color counties by their mean NDVI values.  
3. Overlay county outlines for context.  
4. Add labels so we can identify individual counties on the map.  

This allows us to **see spatial patterns in vegetation health** across administrative units, which is often more useful than just looking at numbers in a table.  


In [14]:
# Convert the county statistics FeatureCollection into an image
ndvi_img = stats_fc.reduceToImage(
    properties=['mean'],
    reducer=ee.Reducer.first()
).selfMask()

# Initialize a map
m3 = geemap.Map()
m3.centerObject(aoi, 8)

# Replace OSM basemap with Google Hybrid
m3.add_tile_layer(
    url="https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}",
    name="Google Hybrid",
    attribution="Google"
)

# Add NDVI image layer (counties colored by mean NDVI)
m3.addLayer(ndvi_img, ndvi_vis, 'NDVI per County')

# Add county outlines on top (black borders, no fill)
m3.addLayer(
    stats_fc.style(**{'color': 'black', 'fillColor': '00000000', 'width': 1}),
    {},
    'County outlines'
)

# Add a colorbar for reference
m3.add_colorbar(ndvi_vis, label='Mean NDVI')

# Add county name labels from the 'NAME' column
m3.add_labels(
    data=stats_fc,
    column='NAME',
    font_size='9pt',
    font_color='black',
)

# Display the map
m3

Map(center=[41.584915675476026, -93.62205399999867], controls=(WidgetControl(options=['position', 'transparent…

### **Step 8. Ranking Counties by NDVI**  

To finish, we will create a chart that shows which counties in our AOI have the **highest mean NDVI values**.  

Steps:  
1. Sort the FeatureCollection by mean NDVI (highest to lowest).  
2. Select the **Top 20 counties** (to keep the chart readable).  
3. Plot a **bar chart** (ColumnChart) with county names on the x-axis and mean NDVI on the y-axis.  

This gives us a clear ranking of counties, which is easier to interpret than a map when comparing many units side by side.  


In [15]:
# Sort counties by the 'mean' NDVI value, descending (highest NDVI first)
stats_fc_sorted = stats_fc.sort('mean', False)

# Select only the Top 20 counties to make the chart easier to read
# (You can adjust the number passed to .limit() to show more or fewer counties)
top_20_counties = stats_fc_sorted.limit(20)

# Create a bar chart:
# - x-axis: county name (NAME property)
# - y-axis: mean NDVI value
# - chart_type: ColumnChart (vertical bars)
fig_bar = geemap.chart.feature_by_feature(
    features=top_20_counties,
    x_property='NAME',
    y_properties=['mean'],
    chart_type='ColumnChart',
    height='400px',
    title='Top 20 Counties by Mean NDVI',
    colors=['green']
)

# Display the chart
fig_bar


# Optional Tasks

### 1. Repeat the **NDVI time series analysis** for the years **2016 through 2024 (inclusive)**.


-  Use the same workflow from this exercise to compute the **monthly median NDVI** for your selected region.  
- Generate a **time-series plot** showing NDVI variation across months and years.  
- Briefly **describe the results** — note any seasonal trends, long-term changes, or unusual patterns you observe.


---

### 2. Reflection Questions
- Review and answer the guiding questions provided in the notebook. **Write your answers (from Task 1, 2 and 3) to a word document and upload it to NTU Learn.**

